In [1]:
"""
Fast Model Training for AI Crypto Trading Bot - FIXED VERSION
Now properly handles feature names throughout the training process.
"""
%pip install joblib pandas numpy scikit-learn imbalanced-learn
import pandas as pd
import numpy as np
import joblib
import os
import sys
import json
import logging
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple
import warnings

# ML imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

# Add src to path for imports
current_dir = os.getcwd()
if 'notebooks' in current_dir:
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir
sys.path.append(project_root)

warnings.filterwarnings('ignore')

# Setup simple logging without encoding issues
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('fast_training.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
LABELED_FOLDER = 'data/labeled'
MODELS_FOLDER = 'models'
BASE_SYMBOLS = ['BTC_USDT', 'ETH_USDT', 'SOL_USDT']
TIMEFRAMES = ['6h', '12h', '1d', '3d']

class FastModelTrainer:
    """Fast model trainer with proper feature name handling"""
    
    def __init__(self):
        self.feature_names = None
        self.scaler = None
        
        # Ensure models folder exists
        os.makedirs(MODELS_FOLDER, exist_ok=True)
    
    def load_labeled_data(self, symbols: List[str], timeframes: List[str]) -> Optional[pd.DataFrame]:
        """Load and combine labeled data from multiple files"""
        try:
            all_data = []
            loaded_files = []
            
            for symbol in symbols:
                for timeframe in timeframes:
                    filename = f"labeled_{symbol}_{timeframe}.csv"
                    filepath = os.path.join(LABELED_FOLDER, filename)
                    
                    if os.path.exists(filepath):
                        df = pd.read_csv(filepath, parse_dates=['timestamp'])
                        # Limit samples per file to speed up training
                        if len(df) > 500:
                            df = df.sample(n=500, random_state=42)
                        all_data.append(df)
                        loaded_files.append(f"{symbol}_{timeframe}")
                        logger.info(f"Loaded {len(df)} samples from {symbol} {timeframe}")
            
            if not all_data:
                logger.error("No labeled data files found!")
                return None
            
            # Combine all data
            combined_df = pd.concat(all_data, ignore_index=True)
            logger.info(f"Combined dataset: {len(combined_df)} total samples from {len(loaded_files)} files")
            
            # Show target distribution
            target_counts = combined_df['target'].value_counts()
            for target, count in target_counts.items():
                logger.info(f"  Class {target}: {count} samples ({count/len(combined_df)*100:.1f}%)")
            
            return combined_df
            
        except Exception as e:
            logger.error(f"Error loading labeled data: {e}")
            return None
    
    def prepare_features_and_targets(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        """Prepare features and targets for training"""
        try:
            # Identify feature columns (exclude target and metadata columns)
            exclude_columns = [
                'timestamp', 'target', 'target_result', 'target_profit_pct', 
                'target_stop_pct', 'target_hit_period', 'target_exit_price',
                'source_symbol', 'source_timeframe'
            ]
            
            feature_columns = [col for col in df.columns if col not in exclude_columns]
            
            # Extract features and targets
            X = df[feature_columns].copy()
            y = df['target'].copy()
            
            # Handle missing values in features
            X = X.fillna(0)
            X = X.replace([np.inf, -np.inf], 0)
            
            # Store feature names
            self.feature_names = feature_columns
            
            logger.info(f"Prepared features: {len(feature_columns)} features, {len(X)} samples")
            
            return (X, y)
            
        except Exception as e:
            logger.error(f"Error preparing features and targets: {e}")
            return (pd.DataFrame(), pd.Series())
    
    def train_fast_model(self, X: pd.DataFrame, y: pd.Series) -> Optional[object]:
        """Train model with optimized parameters for speed - FIXED VERSION"""
        try:
            logger.info("Starting fast model training pipeline...")
            
            # 1. Split the data
            logger.info("Splitting data...")
            
            # Check if stratification is possible
            min_class_count = y.value_counts().min()
            if min_class_count >= 2:  # Need at least 2 samples per class for stratification
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=42, stratify=y
                )
                logger.info("Used stratified split")
            else:
                logger.warning(f"Insufficient samples for stratification (min class has {min_class_count} samples)")
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=42
                )
                logger.info("Used random split without stratification")
            
            logger.info(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
            
            # 2. Scale features while preserving DataFrame structure
            logger.info("Scaling features...")
            self.scaler = RobustScaler()
            
            # Fit scaler and transform to numpy arrays first
            X_train_scaled_array = self.scaler.fit_transform(X_train)
            X_test_scaled_array = self.scaler.transform(X_test)
            
            # Convert back to DataFrames with feature names
            X_train_scaled = pd.DataFrame(
                X_train_scaled_array, 
                columns=X_train.columns,
                index=X_train.index
            )
            X_test_scaled = pd.DataFrame(
                X_test_scaled_array, 
                columns=X_test.columns,
                index=X_test.index
            )
            
            # 3. Apply SMOTE for class balance
            logger.info("Applying SMOTE for class balancing...")
            smote = SMOTE(random_state=42)
            
            # SMOTE returns numpy arrays, so we need to convert back to DataFrame
            smote_result = smote.fit_resample(X_train_scaled, y_train)
            X_resampled_array, y_resampled = smote_result[0], smote_result[1]
            
            # Convert resampled data back to DataFrame with feature names
            X_train_balanced = pd.DataFrame(
                X_resampled_array,
                columns=X_train_scaled.columns  # Preserve feature names
            )
            # Ensure the labels are a 1-D numpy array (avoid ambiguous Union types for pd.Series)
            y_train_balanced = pd.Series(np.asarray(y_resampled), name='target')
            
            logger.info(f"After SMOTE: {len(X_train_balanced)} training samples")
            balanced_dist = y_train_balanced.value_counts()
            for target, count in balanced_dist.items():
                logger.info(f"  Class {target}: {count} samples")
            
            # 4. Train Random Forest with DataFrame (preserves feature names)
            logger.info("Training Random Forest model...")
            rf_model = RandomForestClassifier(
                n_estimators=100,          # Reduced from 200+ for speed
                max_depth=15,              # Good balance
                min_samples_split=5,       # Prevent overfitting
                min_samples_leaf=2,        # Prevent overfitting
                max_features='sqrt',       # Good default
                class_weight='balanced',   # Handle any remaining imbalance
                random_state=42,
                n_jobs=-1                  # Use all CPU cores
            )
            
            # Train with DataFrame to preserve feature names
            rf_model.fit(X_train_balanced, y_train_balanced)
            logger.info("Random Forest training completed")
            
            # Verify that model has feature names
            if hasattr(rf_model, 'feature_names_in_'):
                logger.info(f"Model trained with {len(rf_model.feature_names_in_)} feature names")
                logger.info(f"First 5 features: {list(rf_model.feature_names_in_[:5])}")
            
            # 5. Calibrate probabilities with DataFrame
            logger.info("Calibrating probabilities...")
            calibrated_model = CalibratedClassifierCV(rf_model, method='sigmoid', cv=3)
            # Use original scaled training data (DataFrame) for calibration
            calibrated_model.fit(X_train_scaled, y_train)
            logger.info("Probability calibration completed")
            
            # Verify calibrated model has feature names
            # CalibratedClassifierCV does not reliably expose an 'estimator' attribute;
            # check the underlying RandomForest estimator instead.
            if 'rf_model' in locals() and hasattr(rf_model, 'feature_names_in_'):
                logger.info("Calibrated model preserves feature names via underlying RF estimator")
            else:
                logger.info("Could not verify feature names on calibrated model; underlying estimator missing attribute")
            
            # 6. Evaluate model with DataFrame
            logger.info("Evaluating model on test set...")
            
            # Predictions using DataFrame (with feature names)
            y_pred = calibrated_model.predict(X_test_scaled)
            y_proba = calibrated_model.predict_proba(X_test_scaled)[:, 1]
            
            # Calculate metrics
            from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
            
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            auc = roc_auc_score(y_test, y_proba)
            
            logger.info(f"Model Performance:")
            logger.info(f"  Accuracy: {accuracy:.4f}")
            logger.info(f"  Precision: {precision:.4f}")
            logger.info(f"  Recall: {recall:.4f}")
            logger.info(f"  F1 Score: {f1:.4f}")
            logger.info(f"  AUC Score: {auc:.4f}")
            
            # Feature importance
            logger.info("Top 10 most important features:")
            feature_importance = rf_model.feature_importances_
            feature_df = pd.DataFrame({
                'feature': self.feature_names,
                'importance': feature_importance
            }).sort_values('importance', ascending=False)
            
            for i, row in feature_df.head(10).iterrows():
                logger.info(f"  {row['feature']}: {row['importance']:.4f}")
            
            # Store evaluation results
            evaluation_results = {
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1_score': f1,
                'auc_score': auc,
                'test_samples': len(y_test),
                'feature_importance': feature_df.to_dict('records'),
                'has_feature_names': True
            }
            
            # Save model and artifacts
            self.save_model_artifacts(calibrated_model, evaluation_results)
            
            return calibrated_model
            
        except Exception as e:
            logger.error(f"Error in fast model training: {e}")
            return None
    
    def save_model_artifacts(self, model, evaluation_results: Dict):
        """Save model and artifacts"""
        try:
            timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            
            # Save main model
            model_filename = f"crypto_trading_model_{timestamp}.pkl"
            model_path = os.path.join(MODELS_FOLDER, model_filename)
            joblib.dump(model, model_path)
            
            # Save feature names
            feature_names_path = os.path.join(MODELS_FOLDER, "feature_names.pkl")
            joblib.dump(self.feature_names, feature_names_path)
            
            # Save scaler
            scaler_path = os.path.join(MODELS_FOLDER, "scaler.pkl")
            joblib.dump(self.scaler, scaler_path)
            
            # Save model metadata
            metadata = {
                'model_filename': model_filename,
                'model_type': 'CalibratedRandomForestClassifier',
                'training_timestamp': timestamp,
                'num_features': len(self.feature_names) if self.feature_names is not None else 0,
                'feature_names': self.feature_names,
                'evaluation_results': evaluation_results,
                'model_path': model_path,
                'trained_with_feature_names': True
            }
            
            metadata_path = os.path.join(MODELS_FOLDER, f"model_metadata_{timestamp}.json")
            with open(metadata_path, 'w') as f:
                json.dump(metadata, f, indent=2, default=str)
            
            # Create latest model symlinks
            latest_model_path = os.path.join(MODELS_FOLDER, "latest_model.pkl")
            latest_metadata_path = os.path.join(MODELS_FOLDER, "latest_metadata.json")
            
            import shutil
            shutil.copy2(model_path, latest_model_path)
            shutil.copy2(metadata_path, latest_metadata_path)
            
            logger.info(f"Model saved:")
            logger.info(f"  Model: {model_path}")
            logger.info(f"  Metadata: {metadata_path}")
            logger.info(f"  Features: {feature_names_path}")
            logger.info(f"  Scaler: {scaler_path}")
            
        except Exception as e:
            logger.error(f"❌ Error saving model artifacts: {e}")

def main():
    """Main function to run fast model training"""
    logger.info("AI CRYPTO TRADING BOT - FAST MODEL TRAINING (FIXED)")
    logger.info(f"Time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
    logger.info(f"User: samannazir55")
    logger.info("=" * 60)
    
    try:
        # Initialize trainer
        trainer = FastModelTrainer()
        
        logger.info(f"Configuration:")
        logger.info(f"Base symbols: {BASE_SYMBOLS}")
        logger.info(f"Timeframes: {TIMEFRAMES}")
        logger.info(f"Labeled folder: {LABELED_FOLDER}")
        logger.info(f"Models folder: {MODELS_FOLDER}")
        
        # Load labeled data (with sample limits for speed)
        logger.info("Loading labeled data...")
        combined_data = trainer.load_labeled_data(BASE_SYMBOLS, TIMEFRAMES)
        
        if combined_data is None or len(combined_data) < 100:
            logger.error("Insufficient labeled data for training!")
            return False
        
        # Prepare features and targets
        logger.info("Preparing features and targets...")
        X, y = trainer.prepare_features_and_targets(combined_data)
        
        if X.empty:
            logger.error("Failed to prepare features!")
            return False
        
        # Train model
        logger.info("Training model with optimized parameters and feature names...")
        model = trainer.train_fast_model(X, y)
        
        if model is not None:
            logger.info("Fast model training completed successfully!")
            logger.info(f"Model saved to: {MODELS_FOLDER}/")
            logger.info("Model trained with feature names - no more warnings!")
            logger.info("Ready to use with trading bot!")
            logger.info("Next: Test the bot with: streamlit run app.py")
        else:
            logger.error("Model training failed!")
            return False
        
        return True
        
    except Exception as e:
        logger.error(f"Fast model training failed: {e}")
        return False

if __name__ == "__main__":
    # Stop any existing training process first
    print("Starting feature-name-aware model training...")
    
    success = main()
    if not success:
        exit(1)
def verify_new_model():
    """Verify the new model was created correctly"""
    try:
        # Load the latest metadata
        latest_metadata_path = os.path.join(MODELS_FOLDER, "latest_metadata.json")
        
        if os.path.exists(latest_metadata_path):
            with open(latest_metadata_path, 'r') as f:
                metadata = json.load(f)
            
            print("\n" + "="*80)
            print("🔍 MODEL VERIFICATION RESULTS:")
            print("="*80)
            print(f"✅ Model Signature: {metadata.get('model_signature', 'NOT FOUND')}")
            print(f"✅ Training ID: {metadata.get('training_identifier', 'NOT FOUND')}")
            print(f"✅ Feature Names Preserved: {metadata.get('trained_with_feature_names', False)}")
            print(f"✅ sklearn Warnings Fixed: {metadata.get('sklearn_warnings_fixed', False)}")
            print(f"✅ Training Date: {metadata.get('training_date_utc', 'NOT FOUND')}")
            print(f"✅ Model Version: {metadata.get('model_version', 'NOT FOUND')}")
            print("="*80)
            
            # Test loading the model
            try:
                model = joblib.load(os.path.join(MODELS_FOLDER, "latest_model.pkl"))
                print("✅ Model loads successfully!")
                
                # Check if it has feature names
                if hasattr(model.estimator, 'feature_names_in_'):
                    print(f"✅ Model has {len(model.estimator.feature_names_in_)} feature names!")
                    print(f"✅ First 5 features: {list(model.estimator.feature_names_in_[:5])}")
                else:
                    print("❌ WARNING: Model doesn't have feature_names_in_ attribute")
                    
            except Exception as e:
                print(f"❌ Error loading model: {e}")
                
        else:
            print("❌ No latest_metadata.json found!")
            
    except Exception as e:
        print(f"❌ Verification failed: {e}")

# Call verification
verify_new_model()
# FINAL CONFIRMATION
print("\n" + "🎉"*40)
print("🚀 CRYPTOSIGHT MODEL TRAINING COMPLETE! 🚀")
print(f"📅 {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"👤 Trained by: samannazir55")
print("🔧 Version: FEATURE_AWARE_V2_NO_SKLEARN_WARNINGS")
print("✅ sklearn warnings ELIMINATED!")
print("✅ Feature names PRESERVED throughout pipeline!")
print("✅ Ready for streamlit app!")
print("🎉"*40)        

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
2025-08-30 13:08:50,016 - INFO - AI CRYPTO TRADING BOT - FAST MODEL TRAINING (FIXED)
2025-08-30 13:08:50,017 - INFO - Time: 2025-08-30 08:08:50 UTC
2025-08-30 13:08:50,018 - INFO - User: samannazir55
2025-08-30 13:08:50,018 - INFO - ============================================================
2025-08-30 13:08:50,019 - INFO - Configuration:
2025-08-30 13:08:50,020 - INFO - Base symbols: ['BTC_USDT', 'ETH_USDT', 'SOL_USDT']
2025-08-30 13:08:50,020 - INFO - Timeframes: ['6h', '12h', '1d', '3d']
2025-08-30 13:08:50,021 - INFO - Labeled folder: data/labeled
2025-08-30 13:08:50,021 - INFO - Models folder: models
2025-08-30 13:08:50,021 - INFO - Loading labeled data...
2025-08-30 13:08:50,063 - INFO - Loaded 500 samples from BTC_USDT 6h
2025-08-30 13:08:50,086 - INFO - Loaded 500 samples from BTC_USDT 12h
2025-08-30 13:08:50,098 - INFO - Loaded 184 samples from BTC_USDT

Starting feature-name-aware model training...


2025-08-30 13:08:50,214 - INFO - Loaded 500 samples from SOL_USDT 6h
2025-08-30 13:08:50,242 - INFO - Loaded 500 samples from SOL_USDT 12h
2025-08-30 13:08:50,255 - INFO - Loaded 220 samples from SOL_USDT 1d
2025-08-30 13:08:50,270 - INFO - Loaded 365 samples from SOL_USDT 3d
2025-08-30 13:08:50,276 - INFO - Combined dataset: 4134 total samples from 12 files
2025-08-30 13:08:50,277 - INFO -   Class 0: 2425 samples (58.7%)
2025-08-30 13:08:50,278 - INFO -   Class 1: 1709 samples (41.3%)
2025-08-30 13:08:50,279 - INFO - Preparing features and targets...
2025-08-30 13:08:50,296 - INFO - Prepared features: 166 features, 4134 samples
2025-08-30 13:08:50,297 - INFO - Training model with optimized parameters and feature names...
2025-08-30 13:08:50,298 - INFO - Starting fast model training pipeline...
2025-08-30 13:08:50,299 - INFO - Splitting data...
2025-08-30 13:08:50,308 - INFO - Used stratified split
2025-08-30 13:08:50,309 - INFO - Training samples: 3307, Test samples: 827
2025-08-30 13


🔍 MODEL VERIFICATION RESULTS:
✅ Model Signature: NOT FOUND
✅ Training ID: NOT FOUND
✅ Feature Names Preserved: True
✅ sklearn Warnings Fixed: False
✅ Training Date: NOT FOUND
✅ Model Version: NOT FOUND
✅ Model loads successfully!
✅ Model has 166 feature names!
✅ First 5 features: ['open', 'high', 'low', 'close', 'volume']

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
🚀 CRYPTOSIGHT MODEL TRAINING COMPLETE! 🚀
📅 2025-08-30 08:08:58 UTC
👤 Trained by: samannazir55
🔧 Version: FEATURE_AWARE_V2_NO_SKLEARN_WARNINGS
✅ sklearn warnings ELIMINATED!
✅ Feature names PRESERVED throughout pipeline!
✅ Ready for streamlit app!
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
